In [ ]:
from google.colab import drive
drive.mount('/content/drive')

NLP_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP-Costumer-Classifier/'
print("Drive connected!")
print(f"Path: {NLP_PATH}")

In [ ]:
!pip install transformers torch -q
print("Libraries installed!")

In [ ]:
import pandas as pd
import numpy as np
import pickle
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

print("All libraries imported!")

In [ ]:
with open(NLP_PATH + 'nlp_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test  = data['X_test']
y_train = data['y_train']
y_test  = data['y_test']
le      = data['le']

print("Data loaded successfully!")
print(f"Training samples: {len(X_train):,}")
print(f"Testing samples:  {len(X_test):,}")
print(f"Categories: {list(le.classes_)}")

In [ ]:
# Build TF-IDF Features
print("Building TF-IDF features...")

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")
print("Done!")

In [ ]:
# Train Logistic Regression Baseline
print("Training TF-IDF + Logistic Regression...")
print("Wait 2-3 minutes...")

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)
acc_lr    = accuracy_score(y_test, y_pred_lr)

print("\n" + "="*50)
print(f"TF-IDF + Logistic Regression Accuracy: {acc_lr:.4f} ({acc_lr*100:.1f}%)")
print("="*50)
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

# Save models
joblib.dump(tfidf,    NLP_PATH + 'tfidf.pkl')
joblib.dump(lr_model, NLP_PATH + 'lr_model.pkl')
print("✅ TF-IDF and LR model saved to Drive!")

In [ ]:
# Confusion Matrix
cm   = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix — TF-IDF + LR Model', fontweight='bold', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(NLP_PATH + 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix saved!")

In [ ]:
# Load DistilBERT Tokenizer
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset
import torch

print("Loading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 5000 train / 1000 test — safe limit for free Colab GPU
X_train_bert = X_train[:5000].tolist()
X_test_bert  = X_test[:1000].tolist()
y_train_bert = y_train[:5000].tolist()
y_test_bert  = y_test[:1000].tolist()

print("Tokenizing text...")
train_encodings = tokenizer(
    X_train_bert,
    truncation=True,
    padding=True,
    max_length=128
)
test_encodings = tokenizer(
    X_test_bert,
    truncation=True,
    padding=True,
    max_length=128
)

print("✅ Tokenization complete!")
print(f"Train tokens: {len(train_encodings['input_ids'])}")
print(f"Test tokens:  {len(test_encodings['input_ids'])}")

In [ ]:
# Create PyTorch Dataset
class ComplaintDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ComplaintDataset(train_encodings, y_train_bert)
test_dataset  = ComplaintDataset(test_encodings,  y_test_bert)

print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Test dataset:  {len(test_dataset):,} samples")
print(" Datasets ready!")

In [ ]:
# Train DistilBERT
from transformers import TrainingArguments, Trainer

num_classes = len(le.classes_)
print(f"Number of categories: {num_classes}")

print("Loading DistilBERT model...")
bert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_classes
)

training_args = TrainingArguments(
    output_dir=NLP_PATH + 'bert_results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_dir=NLP_PATH + 'logs',
    load_best_model_at_end=True,
    logging_steps=50
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("="*50)
print("Training DistilBERT...")
print("DO NOT close this tab!")
print("Wait 15-20 minutes...")
print("="*50)

trainer.train()

bert_model.save_pretrained(NLP_PATH + 'bert_model')
tokenizer.save_pretrained(NLP_PATH + 'bert_tokenizer')

print("\n DistilBERT trained and saved to Drive!")

In [ ]:
# Evaluate DistilBERT
from transformers import pipeline

print("Loading trained DistilBERT for evaluation...")

bert_pipeline = pipeline(
    'text-classification',
    model=NLP_PATH + 'bert_model',
    tokenizer=NLP_PATH + 'bert_tokenizer'
)

print("Running predictions on 200 test samples...")
bert_preds = []
for i, text in enumerate(X_test_bert[:200]):
    result = bert_pipeline(text[:512])[0]
    bert_preds.append(int(result['label'].split('_')[1]))
    if (i+1) % 50 == 0:
        print(f"  Done {i+1}/200...")

bert_acc = accuracy_score(y_test_bert[:200], bert_preds)

print("\n" + "="*50)
print("FINAL MODEL COMPARISON")
print("="*50)
print(f"TF-IDF + Logistic Regression : {acc_lr*100:.1f}%")
print(f"DistilBERT                   : {bert_acc*100:.1f}%")
print("="*50)
print("\n WRITE DOWN THESE NUMBERS — USE ON YOUR RESUME!")

In [ ]:
# Model Comparison Chart
models      = ['TF-IDF + LR\n(Baseline)', 'DistilBERT\n(Advanced)']
accuracies  = [acc_lr * 100, bert_acc * 100]
colors      = ['#3498db', '#e74c3c']

plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracies, color=colors, edgecolor='black', width=0.4)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{acc:.1f}%',
        ha='center', va='bottom',
        fontsize=13, fontweight='bold'
    )

plt.title('Model Accuracy Comparison', fontweight='bold', fontsize=14)
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig(NLP_PATH + 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison chart saved!")

In [ ]:
# Save Final Results
results = {
    'tfidf_accuracy':  acc_lr,
    'bert_accuracy':   bert_acc,
    'categories':      list(le.classes_),
    'num_categories':  num_classes
}

with open(NLP_PATH + 'model_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("="*50)
print("✅ NOTEBOOK 3 COMPLETE!")
print("="*50)
print(f"\nFiles saved to Drive:")
print(f"  ✅ tfidf.pkl")
print(f"  ✅ lr_model.pkl")
print(f"  ✅ bert_model/")
print(f"  ✅ bert_tokenizer/")
print(f"  ✅ confusion_matrix.png")
print(f"  ✅ model_comparison.png")
print(f"  ✅ model_results.pkl")
print(f"\nTF-IDF Accuracy : {acc_lr*100:.1f}%")
print(f"DistilBERT Accuracy : {bert_acc*100:.1f}%")
print(f"\nNow open Notebook 4 — Sentiment Analysis")